# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omaradelahmed/fly-rank-internship1/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*
### Finding A -- "What Predicts Growth?" (Logistic Regression, p.29)

**Where does the label come from?** The paper doesn't fully specify how the growing/declining
label was built for this specific model, though the data dictionary elsewhere defines
`trend_direction` from 30-day-vs-previous-30-day impression change (>10% = up, <-10% = down).
If that's the label source here, it's a reasonable, clearly-defined proxy -- but the paper
should state this explicitly next to the finding, not leave the reader to infer it from a
different section.

**Does the validation design carry the claim?** Methodology (p.36) states "Logistic Regression
(80/20 split)" with no mention of grouping by brand, despite the study spanning 57 different
brands. We found in our own Lane 2 work that a plain random split versus a client-grouped
split produces a materially different, more optimistic score on the exact same data and
features -- because rows from the same entity leak shared patterns across the split. The same
risk applies here: if this 80/20 split is a plain random row split, pages from the same brand
could appear in both train and test, and the reported 71% holdout accuracy may be inflated by
that leakage rather than reflecting a truly out-of-sample brand.

We'd also flag that the 71% figure is reported without its base rate. From Finding #1 (p.6),
the growing/declining mix in this portfolio is roughly 62%/38%. A model that always predicts
"growing" would score close to 62% for free -- so the honest read of the 71% figure is closer
to a 9-point lift over a naive baseline, not 71 points of standalone skill. Pairing accuracy
with its base rate (as our own `training-honest-models` workflow requires) would make this
finding easier to trust at face value.

### Finding B -- "What Predicts Health?" (Random Forest, p.27)

**Where does the label come from?** The label is `Health Score`, defined explicitly in the
Methodology section as `Impressions (30pts) + Position (30pts) + CTR (20pts) + Scroll Depth
(20pts)`. This is fully transparent -- no ambiguity about label construction here.

**Does the validation design carry the claim?** This is a different kind of issue than Finding
A, and not one a better split can fix: `Average Position` (43% importance) and `Impressions`
(32% importance) are simultaneously two of the four ingredients used to *compute* the label
itself. High importance is close to guaranteed by construction, regardless of split quality.
To the paper's credit, it states this outright: *"the target itself is partly constructed from
some of these inputs, so importance is descriptive rather than causal"* -- which is exactly the
kind of honest disclosure we tried to practice with our own `imp_label_window` leakage trap in
`w03_data_contract.ipynb`. We'd only suggest going one step further: since 75% of the model's
importance is structurally guaranteed, the remaining features (Scroll Depth 15%, CTR 8%) are
the only ones that carry real predictive information, and the write-up could say that
explicitly rather than leaving all ten features in one ranked list.

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

pd.set_option('display.max_columns', None)

# Loaded from Week-4's output -- run w04_baseline_score.ipynb first in this Colab session
df = pd.read_csv('work/outputs/baseline_action_score.csv')

exclude_cols = [
    'client_hash_id', 'content_hash_id', 'is_declining',
    'stale_visible_page', 'declining_with_demand', 'thin_content', 'page_one_decay_risk',
    'baseline_score', 'any_rule_triggered', 'reason_codes', 'rank',
    'imp_label_window'  # label-derived (future window) -- must never be a feature
]
feature_cols = [c for c in df.columns if c not in exclude_cols]
X = df[feature_cols].copy()
y = df['is_declining'].copy()
X_encoded = pd.get_dummies(X, columns=['content_type', 'main_intent'], drop_first=True)

print(f"Loaded {len(df):,} rows, {X_encoded.shape[1]} encoded features")

# --- Supporting check for Finding A: the base rate the paper's 71% figure omits ---
growing, declining = 74_800, 45_600  # paper's Finding #1, p.6
base_rate = growing / (growing + declining)
print(f"\nFinding A -- paper's implied base rate (growing share): {base_rate:.1%}")
print(f"Paper reports 71% holdout accuracy -> real lift over a naive baseline is roughly "
      f"{71 - base_rate*100:.0f} points, not 71.")

# --- Supporting check for Finding B: how much RF importance is structurally guaranteed ---
structural_share = 43 + 32
print(f"\nFinding B -- share of RF importance from features that are also label ingredients: {structural_share}%")
print(f"Only the remaining {100-structural_share}% (Scroll Depth, CTR, etc.) carries information "
      f"beyond what the label's own formula already guarantees.")

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# BEFORE: a plain random row split (what the paper's methodology describes -- "80/20 split",
# no grouping mentioned, despite 57 different brands in the study)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf_random.fit(X_tr_r, y_tr_r)
auc_random = roc_auc_score(y_te_r, rf_random.predict_proba(X_te_r)[:, 1])

# AFTER: client-holdout (grouped) split -- what we actually used for our Week-5 headline numbers
groups = df['client_hash_id']
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X_encoded, y, groups=groups))
X_tr_g, X_te_g = X_encoded.iloc[train_idx], X_encoded.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf_grouped.fit(X_tr_g, y_tr_g)
auc_grouped = roc_auc_score(y_te_g, rf_grouped.predict_proba(X_te_g)[:, 1])

print(f"BEFORE -- plain random row split (80/20, no grouping):        ROC-AUC = {auc_random:.3f}")
print(f"AFTER  -- client-holdout split (grouped by client_hash_id):   ROC-AUC = {auc_grouped:.3f}")
print(f"Gap: {auc_random - auc_grouped:+.3f}  -- a plain split can look better for the wrong "
      f"reason, the exact risk we flagged in the paper's Finding A.")

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# Same leakage hunt as w03, now on our FINAL Lane-2 feature set (not the toy 15/15-day slice).
# imp_label_window's raw value is still sitting in df (we only excluded it from feature_cols,
# not from df itself) -- no need to reconnect to HF or refetch it.

X_clean = X_encoded.copy()
X_leaky = X_encoded.copy()
X_leaky['imp_label_window'] = df['imp_label_window'].values

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_clean, y, test_size=0.25, random_state=42, stratify=y)
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)

rf_clean = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=20, random_state=42, n_jobs=-1).fit(X_tr_c, y_tr_c)
rf_leaky = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=20, random_state=42, n_jobs=-1).fit(X_tr_l, y_tr_l)

auc_clean = roc_auc_score(y_te_c, rf_clean.predict_proba(X_te_c)[:, 1])
auc_leaky = roc_auc_score(y_te_l, rf_leaky.predict_proba(X_te_l)[:, 1])

print(f"Honest ROC-AUC (final feature set, no label-derived columns): {auc_clean:.3f}")
print(f"Leaky ROC-AUC (imp_label_window deliberately added):          {auc_leaky:.3f}")
print(f"Jump from the leak: {auc_leaky - auc_clean:+.3f}")

del X_leaky, rf_leaky
print(f"\nKept score -- honest ROC-AUC = {auc_clean:.3f}. No label-derived columns in the final feature set.")

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
original_claim = (
    "Our Random Forest model beats FlyRank's baseline rules and proves which content will decline."
)

rewritten_claim = (
    "On a held-out set of clients the model has never scored before, Random Forest ranks "
    "content by decline risk with 98.0% precision in the top 50 candidates (vs a 56-58% base "
    "rate of actually-declining pages in that same test set) -- matching our leakage-audited "
    "rule-based baseline at that same cutoff. This is an observed, out-of-sample ranking result, "
    "not a causal claim: the model flags candidates for human review, it does not prove why a "
    "page will decline."
)

print("ORIGINAL (overclaimed):")
print(original_claim)
print("\nREWRITTEN (safe language -- observed / measured / decision-support):")
print(rewritten_claim)

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.